In [17]:
import json
import os
import pandas as pd


In [18]:
CURR_PATH = !pwd
TMP_DF_PATH = os.path.join(str(CURR_PATH[0]), '..', '..', 'tmp', '01-ingest-and-transform-inventory', '2025-27-10_inventory_tmp.csv')
TMP_DF_PATH

'/Users/luisenrique/Documents/network-inventory-cleaning-and-validation/src/fastapi/../../tmp/01-ingest-and-transform-inventory/2025-27-10_inventory_tmp.csv'

In [30]:
df_tmp = pd.read_csv(TMP_DF_PATH, index_col=0)
df_tmp

,ip,ip_valid,ip_type,subnet_cidr,hostname,hostname_valid,fqdn,fqdn_consistent,reverse_ptr,mac,mac_valid,owner,owner_email,owner_team,device_type,device_type_confidence,site,site_normalized,source_row_id,normalization_steps
0,192.168.10.5,True,private_rfc1918,192.168.10.0/24,host01,True,NaN,inconsistent,5.10.168.192.in-addr.arpa,aa:bb:cc:dd:ee:ff,True,priya,priya@corp.example.com,platform,server,100,BLR Campus,blr-campus,1,"[{'field': 'ip', 'from': '192.168.010.005', 't..."
1,NaN,False,invalid,NaN,host-02,True,host-02.local,inconsistent,NaN,11:22:33:44:55:66,True,ops,NaN,NaN,unknown,0,HQ Bldg 1,hq-bldg-1,2,"[{'field': 'ip', 'from': '10.0.1.300', 'to': N..."
2,NaN,False,invalid,NaN,host03,True,NaN,inconsistent,NaN,aa:bb:cc:dd:ee:ff,True,NaN,jane@corp.example.com,NaN,switch,100,HQ-BUILDING-1,hq-bldg-1,3,"[{'field': 'ip', 'from': '10.0.1', 'to': None,..."
3,NaN,False,invalid,NaN,printer-01,True,NaN,inconsistent,NaN,00:11:22:33:44:55,True,facilities,NaN,NaN,printer,100,HQ,hq,4,"[{'field': 'ip', 'from': '10.0.1.1.2', 'to': N..."
4,NaN,False,invalid,NaN,iot-cam01,True,NaN,inconsistent,NaN,00:aa:bb:cc:dd:ee,True,sec,NaN,NaN,unknown,0,Lab-1,lab-1,5,"[{'field': 'ip', 'from': 'fe80::1%eth0', 'to':..."
5,127.0.0.1,True,loopback,127.0.0.0/8,local-test,True,NaN,inconsistent,1.0.0.127.in-addr.arpa,NaN,False,NaN,NaN,NaN,unknown,0,NaN,unknown,6,"[{'field': 'ip', 'from': '127.0.0.1', 'to': '1..."
6,169.254.10.20,True,link_local_apipa,169.254.0.0/16,host-apipa,True,NaN,inconsistent,20.10.254.169.in-addr.arpa,NaN,False,NaN,NaN,NaN,unknown,0,NaN,unknown,7,"[{'field': 'ip', 'from': '169.254.10.20', 'to'..."
7,10.10.10.10,True,private_rfc1918,10.0.0.0/8,srv-10,True,NaN,inconsistent,10.10.10.10.in-addr.arpa,NaN,False,platform,NaN,NaN,server,100,BLR campus,blr-campus,8,"[{'field': 'ip', 'from': ' 10.10.10.10 ', 't..."
8,NaN,False,invalid,NaN,badhost,True,NaN,inconsistent,NaN,NaN,False,NaN,NaN,NaN,unknown,0,NaN,unknown,9,"[{'field': 'ip', 'from': 'abc.def.ghi.jkl', 't..."
9,NaN,False,invalid,NaN,neg,True,NaN,inconsistent,NaN,NaN,False,NaN,NaN,NaN,unknown,0,NaN,unknown,10,"[{'field': 'ip', 'from': '192.168.1.-1', 'to':..."


In [31]:
df_response = pd.read_json("response.json")
df_response

,owner,owner_team,owner_email,confidence_percentage
0,priya,platform,priya@corp.example.com,90
1,None,ops,None,95
2,jane,None,jane@corp.example.com,70
3,None,facilities,None,95
4,None,sec,None,90
5,None,None,None,100
6,None,None,None,100
7,None,platform,None,95
8,None,None,None,100
9,None,None,None,100


In [34]:
high_confidence_indeces = df_response[df_response['confidence_percentage'] >= 70].index
high_confidence_indeces

Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14], dtype='int64')

In [ ]:
# KEEPING OBSERVATIONS WITH A CONFIDENE >= 70.
df_tmp.loc[high_confidence_indeces, ['owner', 'owner_email', 'owner_team']] = df_response.loc[high_confidence_indeces, ['owner', 'owner_email', 'owner_team']] 
df_tmp

,ip,ip_valid,ip_type,subnet_cidr,hostname,hostname_valid,fqdn,fqdn_consistent,reverse_ptr,mac,mac_valid,owner,owner_email,owner_team,device_type,device_type_confidence,site,site_normalized,source_row_id,normalization_steps
0,192.168.10.5,True,private_rfc1918,192.168.10.0/24,host01,True,NaN,inconsistent,5.10.168.192.in-addr.arpa,aa:bb:cc:dd:ee:ff,True,priya,priya@corp.example.com,platform,server,100,BLR Campus,blr-campus,1,"[{'field': 'ip', 'from': '192.168.010.005', 't..."
1,NaN,False,invalid,NaN,host-02,True,host-02.local,inconsistent,NaN,11:22:33:44:55:66,True,None,None,ops,unknown,0,HQ Bldg 1,hq-bldg-1,2,"[{'field': 'ip', 'from': '10.0.1.300', 'to': N..."
2,NaN,False,invalid,NaN,host03,True,NaN,inconsistent,NaN,aa:bb:cc:dd:ee:ff,True,jane,jane@corp.example.com,None,switch,100,HQ-BUILDING-1,hq-bldg-1,3,"[{'field': 'ip', 'from': '10.0.1', 'to': None,..."
3,NaN,False,invalid,NaN,printer-01,True,NaN,inconsistent,NaN,00:11:22:33:44:55,True,None,None,facilities,printer,100,HQ,hq,4,"[{'field': 'ip', 'from': '10.0.1.1.2', 'to': N..."
4,NaN,False,invalid,NaN,iot-cam01,True,NaN,inconsistent,NaN,00:aa:bb:cc:dd:ee,True,None,None,sec,unknown,0,Lab-1,lab-1,5,"[{'field': 'ip', 'from': 'fe80::1%eth0', 'to':..."
5,127.0.0.1,True,loopback,127.0.0.0/8,local-test,True,NaN,inconsistent,1.0.0.127.in-addr.arpa,NaN,False,None,None,None,unknown,0,NaN,unknown,6,"[{'field': 'ip', 'from': '127.0.0.1', 'to': '1..."
6,169.254.10.20,True,link_local_apipa,169.254.0.0/16,host-apipa,True,NaN,inconsistent,20.10.254.169.in-addr.arpa,NaN,False,None,None,None,unknown,0,NaN,unknown,7,"[{'field': 'ip', 'from': '169.254.10.20', 'to'..."
7,10.10.10.10,True,private_rfc1918,10.0.0.0/8,srv-10,True,NaN,inconsistent,10.10.10.10.in-addr.arpa,NaN,False,None,None,platform,server,100,BLR campus,blr-campus,8,"[{'field': 'ip', 'from': ' 10.10.10.10 ', 't..."
8,NaN,False,invalid,NaN,badhost,True,NaN,inconsistent,NaN,NaN,False,None,None,None,unknown,0,NaN,unknown,9,"[{'field': 'ip', 'from': 'abc.def.ghi.jkl', 't..."
9,NaN,False,invalid,NaN,neg,True,NaN,inconsistent,NaN,NaN,False,None,None,None,unknown,0,NaN,unknown,10,"[{'field': 'ip', 'from': '192.168.1.-1', 'to':..."
